# LLM advertising spans

Reads each raw transcript CSV with a local instruction-tuned LLM and returns advertising spans as half-open, 0-based transcript line ranges: `[start, end)`. The model sees lines formatted as `[index] text`, so the labels can be mapped back to the original `start` and `end` timestamps without changing the transcript.

Each episode gets a JSON file under `out/llm_ad_spans/` with the same relative path as the input and a `.json` suffix. Long transcripts are processed in overlapping chunks; duplicate and adjacent spans are merged.

This notebook uses the repository's local `vllm` setup. Run it from the repo root and point `INPUT_ROOT` at the raw transcript CSV directory.

In [ ]:
import csv
import json
import re
from pathlib import Path

import numpy as np
import plotly.graph_objects as go
from vllm import LLM, SamplingParams

from hot_topic.vllm_utils import make_structured_outputs_params

MODEL_NAME = "openai/gpt-oss-120b"
# INPUT_ROOT = Path("out/resegmented")
INPUT_ROOT = Path("out/whisper_segmented")
OUTPUT_ROOT = Path("out/llm_ad_spans")
FIG_ROOT = Path('out/llm_ad_plots')
N_EPISODES = None  # set to an int to process only the N largest transcripts
EPISODE_FILES = None  # or a list of Paths under INPUT_ROOT

MAX_MODEL_LEN = 32768
MAX_NEW_TOKENS = 2048
MAX_LINES_PER_REQUEST = 1200
CHUNK_OVERLAP = 24

AD_SPAN_SCHEMA = {
    "type": "object",
    "properties": {
        "spans": {
            "type": "array",
            "items": {
                "type": "object",
                "properties": {
                    "start": {"type": "integer", "minimum": 0},
                    "end": {"type": "integer", "minimum": 1},
                    "reason": {"type": "string"},
                },
                "required": ["start", "end"],
                "additionalProperties": False,
            },
        },
    },
    "required": ["spans"],
    "additionalProperties": False,
}

AD_SPAN_OUTPUTS = make_structured_outputs_params(MODEL_NAME, json_schema=AD_SPAN_SCHEMA)

In [ ]:
llm = LLM(model=MODEL_NAME, max_model_len=MAX_MODEL_LEN, max_num_seqs=1, gpu_memory_utilization=0.95)

## Prompt and response parsing

The model is constrained to return JSON matching `AD_SPAN_SCHEMA`, in the form `{"spans": [{"start": 12, "end": 18, "reason": "..."}]}`. `start` is inclusive and `end` is exclusive, so this example labels lines 12 through 17. Parsing still tolerates surrounding model text, while validation rejects out-of-range or malformed spans.

In [ ]:
SYSTEM_PROMPT = """You identify advertising in podcast transcripts.
Return one JSON object matching the provided schema, with a `spans` array. Each span must contain integer `start` and `end` fields and may contain a short `reason` field.
Spans use half-open intervals: `start` is inclusive and `end` is exclusive. Indices refer to the 0-based line numbers supplied in the transcript.
"""

USER_PROMPT = """Read the transcript below and identify every contiguous range of transcript lines that is advertising. Include sponsor reads, product or service promotions, affiliate messages, fundraising or membership appeals, and promotions for the podcast or its network. Do not label ordinary discussion of a product, news, or a guest's work unless it is promotional.

Return JSON matching the schema. Use half-open spans: `start` is the first advertising line and `end` is one past the last advertising line. For example, `{"start": 0, "end": 3}` labels lines 0, 1, and 2.

Use the supplied line indices exactly. If there is no advertising, return {"spans": []}. Do not include any text outside the JSON object.

TRANSCRIPT:
"""


def episode_paths(root, files, n):
    root = Path(root).resolve()
    if not root.is_dir():
        raise FileNotFoundError(f"Transcript directory does not exist: {root}")
    if files:
        paths = []
        for path in files:
            resolved = Path(path).resolve()
            if not resolved.is_relative_to(root):
                raise ValueError(f"input file is outside INPUT_ROOT: {path}")
            paths.append(resolved)
    else:
        paths = list(root.glob("**/*.csv"))
    paths = sorted(paths, key=lambda path: path.stat().st_size, reverse=True)
    return paths if n is None else paths[:n]


def read_transcript(path):
    with open(path, newline="", encoding="utf-8") as source:
        rows = list(csv.DictReader(source))
    if rows:
        missing = {"start", "end", "text"} - set(rows[0])
        if missing:
            raise ValueError(f"{path} is missing columns: {sorted(missing)}")
    return rows


def format_transcript(rows, offset=0):
    return "\n".join(f"[{offset + index}] {row['text'].strip()}" for index, row in enumerate(rows))


def extract_json_object(text):
    decoder = json.JSONDecoder()
    for match in re.finditer(r"\{", text):
        try:
            value, _ = decoder.raw_decode(text[match.start():])
        except json.JSONDecodeError:
            continue
        if isinstance(value, dict) and "spans" in value:
            return value
    raise ValueError(f"LLM response did not contain a spans JSON object: {text[:500]!r}")


def validate_spans(payload, line_count, offset=0):
    if not isinstance(payload, dict) or not isinstance(payload.get("spans"), list):
        raise ValueError("LLM response must contain a spans array")
    validated = []
    lower = offset
    upper = offset + line_count
    for span in payload["spans"]:
        if not isinstance(span, dict) or not isinstance(span.get("start"), int) or not isinstance(span.get("end"), int):
            raise ValueError(f"Malformed span: {span!r}")
        start, end = span["start"], span["end"]
        if not lower <= start < end <= upper:
            raise ValueError(f"Span {span!r} is outside half-open line range [{lower}, {upper})")
        validated.append({"start": start, "end": end, "reason": str(span.get("reason", ""))})
    return validated


def merge_spans(spans):
    merged = []
    for span in sorted(spans, key=lambda item: (item["start"], item["end"])):
        if merged and span["start"] <= merged[-1]["end"]:
            merged[-1]["end"] = max(merged[-1]["end"], span["end"])
            if span.get("reason") and span["reason"] not in merged[-1].get("reason", ""):
                merged[-1]["reason"] = f"{merged[-1].get('reason', '')}; {span['reason']}".strip("; ")
        else:
            merged.append(dict(span))
    return merged

## Label episodes

In [ ]:
def label_transcripts(input_path):
    input_path = Path(input_path)
    rows = read_transcript(input_path)
    all_spans = []
    if rows:
        step = max(1, MAX_LINES_PER_REQUEST - CHUNK_OVERLAP)
        chunks = [(offset, rows[offset:offset + MAX_LINES_PER_REQUEST]) for offset in range(0, len(rows), step)]
        conversations = [
            [
                {"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user", "content": USER_PROMPT + format_transcript(chunk, offset)},
            ]
            for offset, chunk in chunks
        ]
        sampling_params = SamplingParams(
            temperature=0.0,
            max_tokens=MAX_NEW_TOKENS,
            structured_outputs=AD_SPAN_OUTPUTS,
        )
        outputs = llm.chat(conversations, sampling_params=sampling_params, use_tqdm=True)
        for (offset, chunk), output in zip(chunks, outputs):
            payload = extract_json_object(output.outputs[0].text)
            all_spans.extend(validate_spans(payload, len(chunk), offset))
    return {
        "source": str(input_path),
        "line_count": len(rows),
        "spans": merge_spans(all_spans),
    }


def extract_ad_text(transcript_path, json_output):
    rows = read_transcript(transcript_path)
    spans = json_output.get("spans", [])
    ad_text = []
    for span in spans:
        start, end = span["start"], span["end"]
        if not 0 <= start < end <= len(rows):
            raise ValueError(f"Span {span!r} is outside transcript line range [0, {len(rows)})")
        ad_text.append(" ".join(row["text"].strip() for row in rows[start:end]))
    return ad_text

## Plot ad spans

`plot_ad_spans` accepts a label JSON path, the parsed label dictionary, or a list of span dictionaries. Spans use half-open `[start, end)` indices, and the plot shades from the start of line `start` through the end of line `end - 1`. Timestamp parsing follows the same `clock`, seconds, centiseconds, and milliseconds conventions as the classifier notebook.

In [ ]:
def _clock_to_seconds(text):
    total = 0.0
    for part in str(text).strip().split(":"):
        total = total * 60 + float(part)
    return total


def timestamps_to_seconds(values, timestamp_unit="auto"):
    texts = [str(value).strip() for value in values]
    if timestamp_unit == "auto" and any(":" in text for text in texts):
        return [_clock_to_seconds(text) for text in texts], "clock"
    if timestamp_unit == "clock":
        return [_clock_to_seconds(text) for text in texts], "clock"
    numbers = [float(text) for text in texts]
    divisors = {"s": 1.0, "cs": 100.0, "ms": 1000.0}
    if timestamp_unit in divisors:
        return [number / divisors[timestamp_unit] for number in numbers], timestamp_unit
    if timestamp_unit != "auto":
        raise ValueError("timestamp_unit must be 'auto', 'clock', 's', 'cs', or 'ms'")
    gaps = [b - a for a, b in zip(numbers, numbers[1:]) if b > a]
    median_gap = float(np.median(gaps)) if gaps else 0.0
    if median_gap >= 500:
        return [number / 1000.0 for number in numbers], "ms"
    if median_gap >= 50:
        return [number / 100.0 for number in numbers], "cs"
    return numbers, "s"


def load_ad_spans(labels):
    if isinstance(labels, (str, Path)):
        with open(labels, encoding="utf-8") as source:
            labels = json.load(source)
    if isinstance(labels, dict):
        labels = labels.get("spans", [])
    labels = [dict(start=span["start"], end=span["end"], reason=str(span.get("reason", ""))) for span in labels]
    return merge_spans(labels)


def plot_ad_spans(transcript_csv, labels, timestamp_unit="auto"):
    rows = read_transcript(transcript_csv)
    if not rows:
        raise ValueError(f"{transcript_csv} has no transcript lines")
    seconds, unit = timestamps_to_seconds([row["start"] for row in rows], timestamp_unit)
    ends, _ = timestamps_to_seconds([row["end"] for row in rows], timestamp_unit)
    minutes = np.asarray(seconds) / 60.0
    spans = load_ad_spans(labels)
    hover_text = [row["text"].strip().replace("\n", " ")[:500] for row in rows]
    figure = go.Figure()
    figure.add_trace(go.Scatter(x=minutes, y=np.zeros(len(rows)), mode="markers", marker=dict(size=3, color="#2f4858"), name="Transcript lines", text=hover_text, customdata=np.arange(len(rows)), hovertemplate="line %{customdata}<br>%{x:.2f} min<br>%{text}<extra></extra>"))
    for index, span in enumerate(spans):
        start, end = span["start"], span["end"]
        if not 0 <= start < end <= len(rows):
            raise ValueError(f"Span {span!r} is outside transcript line range [0, {len(rows)})")
        figure.add_vrect(x0=minutes[start], x1=ends[end - 1] / 60.0, fillcolor="#e76f51", opacity=0.28, line_width=0, annotation_text=f"ad [{start}, {end})", annotation_position="top left" if index % 2 == 0 else "top right")
    figure.update_layout(title=f"{Path(transcript_csv).name}: advertising spans", xaxis_title=f"Minutes into episode ({unit})", yaxis=dict(visible=False, range=[-1, 1]), height=360, template="plotly_white", showlegend=False)
    print(f"{transcript_csv}: {len(rows)} lines, {len(spans)} advertising spans, timestamps read as {unit}")
    return figure

In [ ]:
# Example after labeling: choose a transcript and its matching JSON label file.
transcript_paths = sorted(Path(INPUT_ROOT).glob("**/*.csv"))

In [ ]:
transcript_path = transcript_paths[0]
json_output = label_transcripts(transcript_path)

In [ ]:
json_output

In [ ]:
import os
csv_relpath = transcript_path.relative_to(INPUT_ROOT)
label_path = Path(OUTPUT_ROOT) / csv_relpath.with_suffix(".json")
os.makedirs(os.path.dirname(label_path), exist_ok=True)
with open(label_path, 'w') as w:
    json.dump(json_output, w, indent=2)


In [ ]:
fig = plot_ad_spans(transcript_path, label_path)
fig

In [ ]:
plot_path = Path(FIG_ROOT) / csv_relpath.with_suffix('.png')
os.makedirs(os.path.dirname(plot_path), exist_ok=True)
fig.write_image(plot_path)

In [ ]:
plot_path = Path(FIG_ROOT) / csv_relpath.with_suffix('.html')
os.makedirs(os.path.dirname(plot_path), exist_ok=True)
fig.write_html(plot_path)

In [ ]:
ad_texts = extract_ad_text(transcript_path, json_output)

In [ ]:
for t in ad_texts:
    print(t)